# Feature scaling: practice

**Author:** [firstname] [lastname]

COSC 3430 Artificial Intelligence | Fall 26 | Augustana U.

End-to-End ML Project, part 6: feature scaling.

Run the setup cell first. It loads the housing data and rebuilds
`housing`, `housing_labels` and `housing_num` from part 5.

In [ ]:
# Setup: load the housing data and rebuild the training set from part 5
# data frames and arrays
import pandas as pd
import numpy as np
# train/test split
from sklearn.model_selection import train_test_split

# load the California housing data
housing_full = pd.read_csv("https://tinyurl.com/housing-csv")
# income categories for stratified sampling
housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5])
# 80/20 split that keeps the income mix in both sets
strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2,
    stratify=housing_full["income_cat"],
    random_state=42)
# drop the helper column from both sets
for df in (strat_train_set, strat_test_set):
    df.drop("income_cat", axis=1, inplace=True)
# predictors: the training set without the target
housing = strat_train_set.drop("median_house_value", axis=1)
# labels: a copy of the target column
housing_labels = strat_train_set["median_house_value"].copy()
# numerical features only
housing_num = housing.select_dtypes(include=[np.number])
# check the dimensions
print(housing.shape, housing_num.shape)

Expected output:

```
(16512, 9) (16512, 8)
```

## Practice: does scaling help a model?

The cell below trains kNN to predict `median_house_value` from five raw
features and prints the error (RMSE) on the test set. Run it.

In [ ]:
# Practice starter: kNN on five raw features
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error

# features, training and test data
cols = ["longitude", "latitude", "total_rooms", "population", "median_income"]
X_train = housing_num[cols]
X_test = strat_test_set[cols]
y_test = strat_test_set["median_house_value"]

# train on the raw features, measure the error on the test set
knn = KNeighborsRegressor()
knn.fit(X_train, housing_labels)
print("RMSE, raw features:", round(root_mean_squared_error(y_test, knn.predict(X_test))))

Is that error good or bad? Compare it with a "model" that ignores the features
and always predicts the mean price of the training set. Run the cell below.

In [ ]:
# Baseline: always predict the mean price of the training set
baseline = np.full(len(y_test), housing_labels.mean())
print("RMSE, always the mean:", round(root_mean_squared_error(y_test, baseline)))

Your turn:

1. Standardize the features with `StandardScaler`.
2. Fit the scaler on `X_train` **only**, then transform `X_train` and `X_test`.
3. Train `knn` again on the scaled data and print the new error.

*Hint:* like the `SimpleImputer` in part 5: `scaler.fit(...)` once, then
`scaler.transform(...)` for each set.

Which error is smaller - and why?

**Why the test set?** You were told never to look at the test data before the
final test. Here, the scaler and kNN are fit on the training data only, so nothing
from the test set goes into a model. Comparing two models on the test set is still
a choice made with test data - we do it only to see how big the effect of scaling
is. In the project, you compare models on a validation set or with
cross-validation (step 6: select a model and train it), and use the test set
once, at the end, for the model you chose.